# 🫁 ResNet-50 para Radiografías de Tórax (CXR) — Clasificación Multilabel con Co-ocurrencia
## TFM: Sistema de Apoyo a la Decisión Clínica Multimodal — Módulo de Imagen
### Universidad de Salamanca · Máster en Análisis Avanzado de Datos Multivariantes y Big Data

---

## 📋 DESCRIPCIÓN GENERAL DEL NOTEBOOK

Este notebook implementa el **modelo de imagen** del sistema multimodal para predicción de patologías cardiopulmonares sobre el dataset **Symile-MIMIC**.

### ¿Qué hace este código?

1. **Carga y preprocesamiento** de radiografías de tórax (CXR) en formato `.jpg` almacenadas en la estructura de directorios MIMIC-CXR, junto con las variables clínico-demográficas (edad, género, raza, tipo de admisión, orientación de la RX).

2. **Arquitectura ResNet-50 con pesos CheXpert pre-entrenados** (`chexpert-resnet50`) específicamente ajustados sobre radiografías torácicas, con dos extensiones:
   - **Módulo de co-ocurrencia entre etiquetas**: una capa lineal 6×6 aprendible que, antes de la sigmoid final, permite que cada logit "escuche" a los otros 5. Esto modela explícitamente las correlaciones diagnósticas observadas en el EDA (Jaccard Atelectasis–Derrame Pleural = 0.30, Cardiomegalia–Derrame Pleural = 0.27...).
   - **Rama de metadatos clínicos**: embeddings de edad, género, raza, tipo de admisión y vista RX concatenados al embedding de imagen antes de la clasificación. El modelo aprende que la posición AP/PA, el sexo y la raza modulan la probabilidad de cada diagnóstico.

3. **Función de pérdida Masked Binary Cross-Entropy (CheXpert convention)**:  
   - `NaN` → etiqueta no observada → **se enmascara y no contribuye al gradiente**.  
   - `-1` → etiqueta incierta → **se trata como negativo** (política U-zeros) o **positivo** (U-ones) según hiperparámetro.  
   - `0` → negativo confirmado → contribuye normalmente.  
   - `1` → positivo confirmado → contribuye normalmente.  
   - `pos_weight` por etiqueta para compensar el desequilibrio de clases (SPW del EDA).

4. **Augmentación agresiva** de datos de entrenamiento (rotación, flip, brillo, contraste, normalización IMAGENET/CheXpert, ruido gaussiano, elastic transform...) para combatir el sobreajuste dada la heterogeneidad de adquisición en MIMIC.

5. **Validación cruzada anidada (Nested Cross-Validation)**:
   - **Loop externo (K=5)**: estima el rendimiento generalizable real del modelo en datos no vistos durante el ajuste de hiperparámetros.
   - **Loop interno (K=3)**: busca los mejores hiperparámetros (grid search extenso) dentro de cada fold externo.
   - Las particiones respetan las divisiones `train_clean / val_clean / test_clean` del dataset original, y el nested CV se aplica sobre el conjunto de entrenamiento.

6. **Grid de hiperparámetros extenso** diseñado con intuición clínica y técnica: tasas de aprendizaje, schedulers, estrategias de fine-tuning (frozen vs. partial unfreeze), dropout, weight decay, política de incertidumbre (U-zeros vs. U-ones), umbral de clasificación por etiqueta.

7. **Métricas multilabel**: AUC-ROC por etiqueta y macro-promedio, F1-score por etiqueta, Average Precision (AP), calibración (ECE), y análisis de errores por subgrupo (género, raza, vista RX).

8. **Guardado del mejor modelo** con sus pesos, umbrales óptimos por etiqueta y metadatos del experimento.

---

## ⚠️ PARTICULARIDADES Y DECISIONES DE DISEÑO IMPORTANTES

### 1. Etiquetas y convenio CheXpert
- **6 etiquetas**: Atelectasis, Cardiomegaly, Edema, Lung Opacity, No Finding, Pleural Effusion.
- **"No Finding"** solo tiene valores `1` o `NaN` en el dataset — nunca `-1` ni `0`. Se excluye de la pérdida cuando aparece NaN. El modelo aprende que "No Finding=1" implica ausencia de las otras patologías.
- **Atelectasis** tiene 64% de NaN → el masking es crítico. El `pos_weight` es bajo (SPW≈0.04) porque el desequilibrio real entre observables es 24:1.
- **Edema** y **Derrame Pleural** tienen mayor correlación con labs (Spearman ≈ 0.25 con Urea) y co-ocurren frecuentemente → máximo beneficio del módulo de correlación.

### 2. Patrón MNAR y los metadatos
- Los pacientes positivos tienen *menos* missingness en labs, lo que revela que el modelo tabular aprende señales de ausencia. Para el modelo de imagen, incluimos los metadatos clínicos (edad, género, etc.) como rama auxiliar para que el modelo aprenda que ciertas poblaciones tienen mayor riesgo diferencial.

### 3. Pesos pre-entrenados CheXpert
- Se usa el modelo `torchxrayvision` (Cohen et al. 2022) con pesos entrenados sobre CheXpert 14 etiquetas. Este es el estado del arte para radiografías de tórax en PyTorch open-source. Los pesos incluyen normalización específica de CXR (media/std distintas de ImageNet).
- **Estrategia de fine-tuning**: se congela el backbone durante las primeras épocas y luego se descongela progresivamente (layer4 → layer3 → ...) para preservar los features de bajo nivel ya aprendidos sobre millones de CXRs.

### 4. Co-ocurrencia vs. cabezas independientes
- El módulo de correlación añade solo 36 parámetros (6×6) pero permite que el modelo aprenda que "si Cardiomegalia es probable, Derrame Pleural también lo es" directamente desde datos.
- Se pueden comparar ambas arquitecturas con el mismo grid de CV como ablation study para el TFM.

### 5. Nested CV sobre partición train
- El test set (`test_clean.csv`) **nunca se toca** durante el entrenamiento ni el tuning — solo para evaluación final.
- El val set original se puede usar como referencia, pero el nested CV genera sus propias particiones internas.

---


In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 1: INSTALACIÓN DE DEPENDENCIAS
# ══════════════════════════════════════════════════════════════════════════════
# torchxrayvision: librería especializada en CXR con pesos CheXpert pre-entrenados.
# sklearn: para nested CV y métricas multilabel.
# albumentations: augmentación avanzada de imágenes médicas.
# ══════════════════════════════════════════════════════════════════════════════

import subprocess, sys

packages = [
    "torchxrayvision",       # Pesos CheXpert + utilidades CXR
    "albumentations",        # Augmentación de imagen avanzada
    "scikit-learn",          # Nested CV, métricas
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "tqdm",
    "Pillow",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

print("✅ Dependencias instaladas correctamente.")


✅ Dependencias instaladas correctamente.


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 2: IMPORTACIONES
# ══════════════════════════════════════════════════════════════════════════════

import os
import warnings
import random
import json
import copy
import time
from pathlib import Path
from itertools import product as itertools_product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
import torchvision.transforms as transforms
import torchvision.models as tv_models

import torchxrayvision as xrv  # Pesos CheXpert pre-entrenados

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report,
)

warnings.filterwarnings("ignore")

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Usando dispositivo: {DEVICE}")
print(f"   PyTorch: {torch.__version__} | torchxrayvision disponible: {xrv.__version__ if hasattr(xrv, '__version__') else 'OK'}")


c:\TFM\1.Opción - Symile Mimic\tfm_multimodal_clinico\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Usando dispositivo: cpu
   PyTorch: 2.8.0+cpu | torchxrayvision disponible: 1.4.0


In [3]:
# =============================================================================
# CELDA 3 — CONFIGURACIÓN DE RUTAS Y CONSTANTES GLOBALES (VERSIÓN NPY)
# =============================================================================
# Adapta BASE_DATA_DIR a tu entorno local.
# Las CXR están preprocesadas como arrays numpy en data_npy/.
# NO existen imágenes JPG en el árbol de directorios: los paths del CSV
# (columna cxr_path, tipo "files/p17/.../imagen.jpg") son referencias MIMIC
# originales que NO están descargadas. Las imágenes reales son los .npy.
# =============================================================================
 
# ── Rutas base ────────────────────────────────────────────────────────────────
BASE_DATA_DIR = Path(
    r"C:\TFM\1.OPCIÓN - SYMILE MIMIC\SYMILE-MIMIC-A-MULTIMODAL-CLINICAL-DATASET-OF-CHEST-X-RAYS-ELECTROCARDIOGRAMS-AND-BLOOD-LABS-FROM-MIMIC-IV-1.0.0"
)
 
# Subdirectorios con archivos CSV limpios
CSV_DIR    = BASE_DATA_DIR / "data_csv" / "clean"
TRAIN_CSV  = CSV_DIR / "train_clean.csv"
VAL_CSV    = CSV_DIR / "val_clean.csv"
TEST_CSV   = CSV_DIR / "test_clean.csv"
 
# Las CXR están preprocesadas como arrays numpy en data_npy/.
# NO existen imágenes JPG en el árbol de directorios: los paths del CSV
# (columna cxr_path, tipo "files/p17/.../imagen.jpg") son referencias MIMIC
# originales que NO están descargadas. Las imágenes reales son los .npy.
NPY_DIR       = BASE_DATA_DIR / "data_npy"
CXR_NPY_TRAIN = NPY_DIR / "train"    / "cxr_train.npy"
CXR_NPY_VAL   = NPY_DIR / "val"      / "cxr_val.npy"
CXR_NPY_TEST  = NPY_DIR / "test"     / "cxr_test.npy"
# CXR_ROOT ya no se usa para cargar JPGs; se mantiene por compatibilidad
CXR_ROOT = BASE_DATA_DIR
 
# Directorio para guardar checkpoints, métricas y resultados del CV
OUTPUT_DIR = Path("outputs_resnet50_cxr")
OUTPUT_DIR.mkdir(exist_ok=True)
 
# ── Etiquetas multilabel ───────────────────────────────────────────────────────
LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Edema",
    "Lung Opacity",
    "No Finding",
    "Pleural Effusion",
]
N_LABELS = len(LABELS)
 
# ── Pesos positivos por etiqueta para la BCE (del EDA) ────────────────────────
POS_WEIGHTS = {
    "Atelectasis":     0.04,
    "Cardiomegaly":    0.30,
    "Edema":           0.90,
    "Lung Opacity":    0.10,
    "No Finding":      1.00,
    "Pleural Effusion": 0.50,
}
 
# ── Parámetros de imagen ───────────────────────────────────────────────────────
IMG_SIZE = 224
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD  = [0.229, 0.224, 0.225]
 
# ── Variables demográficas incluidas como rama auxiliar ───────────────────────
GENDER_MAP         = {0: 0, 1: 1}
RACE_MAP           = {"UNKNOWN": 0, "WHITE": 1, "BLACK": 2,
                      "ASIAN": 3, "HISPANIC_LATINO": 4}
ADMISSION_MAP      = {"SCHEDULED": 0, "EMERGENCY": 1,
                      "OBSERVATION": 2, "URGENT": 3}
CXR_VIEW_MAP       = {"AP": 0, "PA": 1}
 
print("✅ Configuración de rutas y constantes cargada.")
print(f"   Dataset base : {BASE_DATA_DIR}")
print(f"   Etiquetas    : {LABELS}")
print(f"   POS_WEIGHTS  : {POS_WEIGHTS}")

✅ Configuración de rutas y constantes cargada.
   Dataset base : C:\TFM\1.OPCIÓN - SYMILE MIMIC\SYMILE-MIMIC-A-MULTIMODAL-CLINICAL-DATASET-OF-CHEST-X-RAYS-ELECTROCARDIOGRAMS-AND-BLOOD-LABS-FROM-MIMIC-IV-1.0.0
   Etiquetas    : ['Atelectasis', 'Cardiomegaly', 'Edema', 'Lung Opacity', 'No Finding', 'Pleural Effusion']
   POS_WEIGHTS  : {'Atelectasis': 0.04, 'Cardiomegaly': 0.3, 'Edema': 0.9, 'Lung Opacity': 0.1, 'No Finding': 1.0, 'Pleural Effusion': 0.5}


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 4: GRID DE HIPERPARÁMETROS EXTENSO
# ══════════════════════════════════════════════════════════════════════════════
# Grid diseñado con intuición clínica y técnica para el problema específico:
#
# 1. lr_backbone: la zona 1e-5 – 5e-5 es el rango "safe" para fine-tuning
#    de backbones pre-entrenados en datos médicos. Valores mayores degradan
#    los features CheXpert aprendidos; menores no convergen en tiempo razonable.
#
# 2. lr_head: el clasificador nuevo puede aprender más rápido que el backbone.
#    Ratio 10:1 (head:backbone) es la regla empírica estándar.
#
# 3. scheduler: OneCycleLR es especialmente útil con datasets médicos pequeños
#    (warm-up + annealing en una sola pasada). CosineAnnealingLR es más estable.
#
# 4. unfreeze_epoch: congelar el backbone las primeras épocas y luego 
#    descongelarlo gradualmente evita que el clasificador "destruya" los pesos
#    pre-entrenados antes de aprender la distribución del dataset actual.
#
# 5. uncertainty_policy: "zeros" (U-zeros) trata las etiquetas -1 como negativo;
#    "ones" (U-ones) las trata como positivo. CheXpert original recomienda U-ones
#    para Edema y Atelectasis. Se explora ambas.
#
# 6. use_label_correlation: activa/desactiva el módulo 6×6 de co-ocurrencia.
#    Permite comparación directa (ablation study) para el TFM.
#
# 7. use_meta_branch: activa/desactiva la rama de metadatos (edad, género...).
#    Idem para ablation study.
#
# 8. dropout_rate: regularización en la cabeza clasificadora. Valores altos
#    (0.5) son habituales en clasificación médica con pocos datos.
#
# 9. weight_decay: L2 regularization en el optimizador. Valores en 1e-4 – 1e-3
#    son estándar para fine-tuning médico.
#
# TOTAL de combinaciones: se listan explícitamente; en el CV se muestrea
# un subconjunto aleatorio para evitar tiempos prohibitivos (Random Search).
# ══════════════════════════════════════════════════════════════════════════════

HYPERPARAM_GRID = {
    # ── Tasas de aprendizaje ──────────────────────────────────────────────────
    "lr_backbone": [1e-5, 3e-5, 5e-5],
    # Backbone: valores conservadores para no destruir pesos CheXpert.

    "lr_head": [1e-4, 3e-4, 5e-4],
    # Cabeza clasificadora + módulo correlación + rama metadatos: aprendizaje más rápido.

    # ── Scheduler de tasa de aprendizaje ─────────────────────────────────────
    "scheduler": ["cosine", "onecycle", "plateau"],
    # cosine: decaimiento suave coseno. onecycle: warm-up + annealing en 1 ciclo.
    # plateau: reduce LR cuando val_loss no mejora (más conservador).

    # ── Épocas y estrategia de descongelado ───────────────────────────────────
    "num_epochs": [20, 30],
    # 20 épocas: suficiente para datasets médicos con pre-entrenamiento.
    # 30 épocas: para explorar si hay ganancia adicional con más iteraciones.

    "unfreeze_epoch": [3, 5, 8],
    # Número de épocas con backbone CONGELADO antes de descongelarlo.
    # 3: descongelado rápido. 8: estrategia conservadora, clasificador aprende primero.

    "unfreeze_layers": ["layer4", "layer3_4", "all"],
    # "layer4": solo descongelar el último bloque residual (menos parámetros).
    # "layer3_4": descongelar los dos últimos bloques (más capacidad).
    # "all": descongelar todo el backbone (máxima adaptación al dominio).

    # ── Política de incertidumbre (etiquetas -1 en CheXpert) ─────────────────
    "uncertainty_policy": ["zeros", "ones"],
    # "zeros": etiqueta -1 → tratada como 0 (negativo).
    # "ones":  etiqueta -1 → tratada como 1 (positivo).
    # La política óptima es distinta por etiqueta; aquí se aplica globalmente.

    # ── Regularización ────────────────────────────────────────────────────────
    "dropout_rate": [0.3, 0.5],
    # Dropout en la cabeza clasificadora antes de las capas finales.

    "weight_decay": [1e-4, 5e-4, 1e-3],
    # L2 regularization en Adam/AdamW. Valores mayores = más regularización.

    # ── Tamaño de batch ───────────────────────────────────────────────────────
    "batch_size": [16, 32],
    # 16: mejor para GPUs con poca VRAM o datasets pequeños.
    # 32: mayor estabilidad del gradiente.

    # ── Módulos adicionales ───────────────────────────────────────────────────
    "use_label_correlation": [True, False],
    # True: activa el módulo 6×6 de co-ocurrencia entre etiquetas.
    # False: cabezas de clasificación independientes (baseline).

    "use_meta_branch": [True, False],
    # True: incluye rama de metadatos (edad, género, raza, admisión, vista CXR).
    # False: solo imagen, sin información clínica adicional.

    # ── Umbral de clasificación (post-training) ───────────────────────────────
    "threshold_search": [True],
    # Si True, busca el umbral óptimo por etiqueta en el val set interno
    # del fold (maximizando F1 por etiqueta). Siempre activo.

    # ── Augmentación ──────────────────────────────────────────────────────────
    "augmentation_level": ["moderate", "aggressive"],
    # "moderate": flip + rotación leve + normalización.
    # "aggressive": + brillo/contraste + ruido gaussiano + elastic transform.
}

# ── Número de combinaciones a muestrear (Random Search) ──────────────────────
# Enumerar todas las combinaciones sería >10.000; se muestrea aleatoriamente.
N_RANDOM_CONFIGS = 20  # Aumentar si dispones de más tiempo/GPU.

# Generar todas las combinaciones y muestrear N_RANDOM_CONFIGS
all_keys   = list(HYPERPARAM_GRID.keys())
all_values = list(HYPERPARAM_GRID.values())
all_combos = list(itertools_product(*all_values))
np.random.shuffle(all_combos)
SAMPLED_CONFIGS = [
    dict(zip(all_keys, combo)) for combo in all_combos[:N_RANDOM_CONFIGS]
]

print(f"✅ Grid de hiperparámetros definido.")
print(f"   Combinaciones totales posibles : {len(all_combos):,}")
print(f"   Configuraciones muestreadas    : {len(SAMPLED_CONFIGS)}")
print(f"   Primera configuración de ejemplo:")
for k, v in SAMPLED_CONFIGS[0].items():
    print(f"     {k:30s} = {v}")


✅ Grid de hiperparámetros definido.
   Combinaciones totales posibles : 93,312
   Configuraciones muestreadas    : 20
   Primera configuración de ejemplo:
     lr_backbone                    = 3e-05
     lr_head                        = 0.0001
     scheduler                      = plateau
     num_epochs                     = 20
     unfreeze_epoch                 = 5
     unfreeze_layers                = layer3_4
     uncertainty_policy             = ones
     dropout_rate                   = 0.5
     weight_decay                   = 0.0005
     batch_size                     = 32
     use_label_correlation          = False
     use_meta_branch                = True
     threshold_search               = True
     augmentation_level             = moderate


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 5: PIPELINES DE AUGMENTACIÓN DE IMAGEN
# ══════════════════════════════════════════════════════════════════════════════
# Se definen dos niveles de augmentación para el entrenamiento y uno fijo
# para validación/test (solo normalización, sin transformaciones aleatorias).
#
# ¿Por qué augmentar CXR agresivamente?
# - Las radiografías en MIMIC varían en posicionamiento, exposición y aparato.
# - La orientación AP vs PA introduce diferencias de magnificación cardíaca.
# - El modelo pre-entrenado en CheXpert ya vio augmentaciones; nuestro dataset
#   adicional se beneficia de diversidad artificial para evitar sobreajuste.
#
# Transformaciones por nivel:
# MODERATE:   flip horizontal, rotación ±10°, variación de brillo/contraste.
# AGGRESSIVE: + ruido gaussiano, elastic transform, grid distortion,
#               variación de gamma, coarse dropout (simula obstrucciones).
# TEST:       solo resize + normalización (sin aleatoriedad).
# ══════════════════════════════════════════════════════════════════════════════

def get_augmentation_pipeline(level: str, img_size: int = IMG_SIZE):
    """
    Devuelve un pipeline de albumentations según el nivel de augmentación.
    
    Args:
        level: 'moderate', 'aggressive' o 'test'.
        img_size: tamaño de salida de la imagen cuadrada.
    
    Returns:
        Pipeline albumentations listo para aplicar sobre arrays numpy HxWxC.
    """
    
    # Normalización estándar ImageNet (válida para backbone CheXpert fine-tuned)
    normalize = A.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD)
    to_tensor = ToTensorV2()  # HxWxC numpy → CxHxW tensor
    
    if level == "test":
        # Sin augmentación: solo redimensionar y normalizar.
        return A.Compose([
            A.Resize(img_size, img_size),
            normalize,
            to_tensor,
        ])
    
    elif level == "moderate":
        return A.Compose([
            A.Resize(img_size, img_size),
            
            # Flip horizontal: la simetría derecha-izquierda no afecta al diagnóstico
            # en la mayoría de patologías torácicas (excepto dextrocardia, rara).
            A.HorizontalFlip(p=0.5),
            
            # Rotación pequeña: simula variabilidad en el posicionamiento del paciente.
            A.Rotate(limit=10, p=0.5),
            
            # Cambios de brillo y contraste: simula variabilidad en la exposición.
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
            
            # Ligero desplazamiento y zoom: simula centrado imperfecto del haz.
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=0, p=0.3),
            
            normalize,
            to_tensor,
        ])
    
    elif level == "aggressive":
        return A.Compose([
            A.Resize(img_size, img_size),
            
            # ── Transformaciones geométricas ──────────────────────────────────
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.5),
            A.ShiftScaleRotate(shift_limit=0.07, scale_limit=0.08, rotate_limit=10, p=0.4),
            
            # Elastic transform: deforma suavemente la imagen simulando variaciones
            # anatómicas entre pacientes. Alpha y sigma calibrados para CXR.
            A.ElasticTransform(alpha=80, sigma=8, p=0.3),
            
            # Grid distortion: distorsión de cuadrícula suave.
            A.GridDistortion(num_steps=5, distort_limit=0.15, p=0.2),
            
            # ── Transformaciones de intensidad ────────────────────────────────
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
            
            # Variación de gamma: simula diferencias en la curva de respuesta
            # del detector radiológico.
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),
            
            # Ruido gaussiano: simula el ruido cuántico de la imagen digital.
            A.GaussNoise(var_limit=(5, 25), p=0.3),
            
            # Ligero desenfoque: simula movimiento del paciente durante la exposición.
            A.OneOf([
                A.GaussianBlur(blur_limit=3, p=1.0),
                A.MotionBlur(blur_limit=3, p=1.0),
            ], p=0.2),
            
            # CLAHE: ecualización adaptativa del histograma, mejora el contraste local.
            # Muy usado en preprocesamiento de CXR clínico.
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
            
            # Coarse dropout: elimina regiones aleatorias de la imagen (regularización).
            # Simula opacidades parciales o artefactos de equipamiento.
            A.CoarseDropout(
                max_holes=8, max_height=16, max_width=16,
                min_holes=1, fill_value=0, p=0.2
            ),
            
            normalize,
            to_tensor,
        ])
    
    else:
        raise ValueError(f"Nivel de augmentación desconocido: '{level}'. Usa 'moderate', 'aggressive' o 'test'.")

# Verificación rápida
pipeline_test = get_augmentation_pipeline("test")
pipeline_mod  = get_augmentation_pipeline("moderate")
pipeline_agg  = get_augmentation_pipeline("aggressive")
print("✅ Pipelines de augmentación definidos: test / moderate / aggressive")


✅ Pipelines de augmentación definidos: test / moderate / aggressive


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 6: DATASET PYTORCH PARA CXR MULTILABEL
# ══════════════════════════════════════════════════════════════════════════════
# CXRMultilabelDataset gestiona:
#   - Carga de imagen JPG desde disco y aplicación del pipeline de augmentación.
#   - Codificación de etiquetas multilabel con manejo de NaN y valores -1.
#   - Construcción del vector de metadatos clínicos (edad normalizada + embeddings).
#   - Generación de la máscara binaria para la Masked BCE.
# ══════════════════════════════════════════════════════════════════════════════

class CXRMultilabelDataset(Dataset):
    """
    Dataset PyTorch para radiografías de tórax multilabel.
    
    Etiquetas por muestra: vector float32 de longitud N_LABELS.
    Máscara por muestra:   vector binario float32 de longitud N_LABELS.
        mask[i] = 1 → etiqueta observable → incluir en pérdida.
        mask[i] = 0 → etiqueta NaN         → excluir del gradiente.
    
    Política de incertidumbre (valor -1 en el CSV):
        'zeros': -1 → 0.0 (negativo)
        'ones':  -1 → 1.0 (positivo)
    """
    
    def __init__(
        self,
        dataframe: pd.DataFrame,
        cxr_root: Path,
        augmentation_pipeline,
        uncertainty_policy: str = "zeros",
        labels: list = LABELS,
    ):
        """
        Args:
            dataframe          : DataFrame con columnas cxr_path, Atelectasis, etc.
            cxr_root           : Directorio raíz desde el que se construye la ruta a la imagen.
            augmentation_pipeline : Pipeline de albumentations (train/val/test).
            uncertainty_policy : 'zeros' o 'ones' para tratar etiquetas -1.
            labels             : Lista de nombres de etiquetas (por defecto LABELS global).
        """
        self.df                  = dataframe.reset_index(drop=True)
        self.cxr_root            = cxr_root
        self.transform           = augmentation_pipeline
        self.uncertainty_policy  = uncertainty_policy
        self.labels              = labels
        
        # Precalcular la edad normalizada (min-max sobre el dataset completo).
        # Se normaliza individualmente en el dataset; en producción debería
        # usarse la media/std del TRAIN para no filtrar información del test.
        age_col = self.df["age"].astype(float)
        self.age_min = age_col.min()
        self.age_max = age_col.max()
    
    def __len__(self):
        return len(self.df)
    
    def _encode_labels_and_mask(self, row) -> tuple:
        """
        Convierte las etiquetas del CSV a tensores float32 y genera la máscara.
        
        Convenio:
          NaN  → label=0.0 (arbitrario), mask=0.0 (excluida del gradiente)
          -1   → label según uncertainty_policy, mask=1.0 (incluida)
           0   → label=0.0, mask=1.0
           1   → label=1.0, mask=1.0
        """
        label_vec = np.zeros(len(self.labels), dtype=np.float32)
        mask_vec  = np.zeros(len(self.labels), dtype=np.float32)
        
        for i, lbl in enumerate(self.labels):
            val = row[lbl]
            
            if pd.isna(val):
                # NaN: etiqueta no observada → se excluye de la pérdida
                label_vec[i] = 0.0
                mask_vec[i]  = 0.0
            elif val == -1:
                # Etiqueta incierta (convenio CheXpert): aplicar política
                mask_vec[i]  = 1.0
                label_vec[i] = 1.0 if self.uncertainty_policy == "ones" else 0.0
            else:
                # 0 o 1: observable y determinado
                label_vec[i] = float(val)
                mask_vec[i]  = 1.0
        
        return label_vec, mask_vec
    
    def _encode_metadata(self, row) -> np.ndarray:
        """
        Construye el vector de metadatos clínicos para la rama auxiliar.
        
        Retorna un vector float32 con:
          [0]   edad normalizada al rango [0, 1]
          [1]   género (0=F, 1=M)
          [2-6] raza one-hot (UNKNOWN, WHITE, BLACK, ASIAN, HISPANIC_LATINO)
          [7-10] tipo de admisión one-hot (SCHEDULED, EMERGENCY, OBSERVATION, URGENT)
          [11-12] vista CXR one-hot (AP, PA)
        Total: 13 features
        """
        # Edad normalizada
        age_norm = (float(row["age"]) - self.age_min) / (self.age_max - self.age_min + 1e-8)
        
        # Género
        gender = float(GENDER_MAP.get(row["gender"], 0))
        
        # Raza (one-hot)
        race_vec = np.zeros(len(RACE_MAP), dtype=np.float32)
        race_idx = RACE_MAP.get(str(row["race"]), 0)
        race_vec[race_idx] = 1.0
        
        # Tipo de admisión (one-hot)
        adm_vec = np.zeros(len(ADMISSION_MAP), dtype=np.float32)
        adm_idx = ADMISSION_MAP.get(str(row["admission_type"]), 1)  # default=EMERGENCY
        adm_vec[adm_idx] = 1.0
        
        # Vista CXR (one-hot)
        view_vec = np.zeros(len(CXR_VIEW_MAP), dtype=np.float32)
        view_idx = CXR_VIEW_MAP.get(str(row["cxr_view"]), 0)  # default=AP
        view_vec[view_idx] = 1.0
        
        return np.concatenate([[age_norm, gender], race_vec, adm_vec, view_vec])
    
    def __getitem__(self, idx):
        """
        Devuelve un dict con:
          'image'    : tensor CxHxW float32 (imagen augmentada y normalizada)
          'labels'   : tensor N_LABELS float32 (etiquetas codificadas)
          'mask'     : tensor N_LABELS float32 (máscara de observabilidad)
          'metadata' : tensor 13 float32 (metadatos clínicos)
          'hadm_id'  : int (identificador de la admisión hospitalaria)
        """
        row = self.df.iloc[idx]
        
        # ── 1. Cargar imagen ──────────────────────────────────────────────────
        img_path = self.cxr_root / row["cxr_path"]
        try:
            image = np.array(Image.open(img_path).convert("RGB"))
        except FileNotFoundError:
            # Si la imagen no existe, retornar un tensor de ceros (ocurre raramente
            # por rutas inconsistentes en MIMIC; loguear en producción).
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        
        # ── 2. Augmentación ───────────────────────────────────────────────────
        augmented = self.transform(image=image)
        image_tensor = augmented["image"].float()  # CxHxW, float32
        
        # ── 3. Etiquetas y máscara ────────────────────────────────────────────
        labels, mask = self._encode_labels_and_mask(row)
        
        # ── 4. Metadatos clínicos ─────────────────────────────────────────────
        metadata = self._encode_metadata(row)
        
        return {
            "image"   : image_tensor,
            "labels"  : torch.tensor(labels,   dtype=torch.float32),
            "mask"    : torch.tensor(mask,     dtype=torch.float32),
            "metadata": torch.tensor(metadata, dtype=torch.float32),
            "hadm_id" : int(row["hadm_id"]),
        }


# ── Carga de los DataFrames ────────────────────────────────────────────────────
df_train = pd.read_csv(TRAIN_CSV, sep=";")
df_val   = pd.read_csv(VAL_CSV,   sep=";")
df_test  = pd.read_csv(TEST_CSV,  sep=";")

print(f"✅ DataFrames cargados:")
print(f"   Train : {len(df_train):,} muestras")
print(f"   Val   : {len(df_val):,} muestras")
print(f"   Test  : {len(df_test):,} muestras")

# Vista previa de las etiquetas en train
print("\n── Distribución de etiquetas en TRAIN (etiquetas observadas) ──────────")
for lbl in LABELS:
    vals = df_train[lbl].dropna()
    n_pos = (vals == 1).sum()
    n_neg = (vals == 0).sum()
    n_unc = (vals == -1).sum()
    n_nan = df_train[lbl].isna().sum()
    print(f"  {lbl:20s}: pos={n_pos:4d} | neg={n_neg:4d} | incierto={n_unc:3d} | NaN={n_nan:4d}")


✅ DataFrames cargados:
   Train : 10,000 muestras
   Val   : 750 muestras
   Test  : 464 muestras

── Distribución de etiquetas en TRAIN (etiquetas observadas) ──────────
  Atelectasis         : pos=2867 | neg= 118 | incierto=588 | NaN=6427
  Cardiomegaly        : pos=3346 | neg= 886 | incierto=450 | NaN=5318
  Edema               : pos=2067 | neg=1772 | incierto=834 | NaN=5327
  Lung Opacity        : pos=3108 | neg= 175 | incierto=223 | NaN=6494
  No Finding          : pos=1368 | neg=   0 | incierto=  0 | NaN=8632
  Pleural Effusion    : pos=3618 | neg=1701 | incierto=371 | NaN=4310


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 7: FUNCIÓN DE PÉRDIDA — MASKED BINARY CROSS-ENTROPY
# ══════════════════════════════════════════════════════════════════════════════
# La Masked BCE es el estándar CheXpert para clasificación multilabel con
# etiquetas parcialmente observadas.
#
# Matemáticamente:
#   loss = -1/N_observed * Σ_{i: mask_i=1} [w_i * y_i * log(σ(x_i))
#                                           + (1 - y_i) * log(1 - σ(x_i))]
#
# Donde:
#   x_i    = logit del modelo para la etiqueta i (antes de sigmoid)
#   y_i    = etiqueta binaria codificada (0 o 1)
#   mask_i = 1 si la etiqueta es observable, 0 si es NaN
#   w_i    = pos_weight para compensar desequilibrio de clases
#
# La máscara se aplica ANTES de promediar → las etiquetas NaN no contribuyen
# ni al numerador ni al denominador (diferente a poner 0 en la pérdida, que
# sí contribuiría al denominador y sesgaría el gradiente).
# ══════════════════════════════════════════════════════════════════════════════

class MaskedBCELoss(nn.Module):
    """
    Binary Cross-Entropy con máscara de observabilidad y pesos por clase.
    
    Args:
        pos_weights_dict : dict {label_name: weight} con el peso para la clase positiva.
                           Si None, todos los pesos = 1.0.
        labels           : lista de nombres de etiquetas (para indexar pos_weights_dict).
    """
    
    def __init__(self, pos_weights_dict: dict = None, labels: list = LABELS):
        super().__init__()
        self.labels = labels
        
        if pos_weights_dict is not None:
            weights = torch.tensor(
                [pos_weights_dict.get(lbl, 1.0) for lbl in labels],
                dtype=torch.float32
            )
        else:
            weights = torch.ones(len(labels), dtype=torch.float32)
        
        # Registrar como buffer (se mueve a GPU con .to(device) del módulo padre)
        self.register_buffer("pos_weights", weights)
    
    def forward(self, logits: torch.Tensor, labels: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits : [B, N_LABELS] — salidas crudas del modelo (antes de sigmoid).
            labels : [B, N_LABELS] — etiquetas float32 (0.0 o 1.0).
            mask   : [B, N_LABELS] — float32, 1.0 si observable, 0.0 si NaN.
        
        Returns:
            Escalar: pérdida promedio sobre las observaciones enmascaradas.
        """
        # BCE elemento a elemento sin reducción: [B, N_LABELS]
        bce_elementwise = F.binary_cross_entropy_with_logits(
            logits,
            labels,
            pos_weight=self.pos_weights.to(logits.device),
            reduction="none",
        )
        
        # Aplicar máscara: poner a cero las entradas no observadas
        masked_bce = bce_elementwise * mask
        
        # Promedio sobre las observaciones válidas (suma / número de masks activas)
        n_observed = mask.sum().clamp(min=1e-8)  # Evitar división por cero
        loss = masked_bce.sum() / n_observed
        
        return loss


# Instanciar la función de pérdida con los pesos del EDA
criterion = MaskedBCELoss(pos_weights_dict=POS_WEIGHTS, labels=LABELS)
criterion = criterion.to(DEVICE)

print("✅ MaskedBCELoss instanciada con pos_weights del EDA:")
for lbl, w in POS_WEIGHTS.items():
    print(f"   {lbl:20s} → pos_weight = {w:.2f}")


✅ MaskedBCELoss instanciada con pos_weights del EDA:
   Atelectasis          → pos_weight = 0.04
   Cardiomegaly         → pos_weight = 0.30
   Edema                → pos_weight = 0.90
   Lung Opacity         → pos_weight = 0.10
   No Finding           → pos_weight = 1.00
   Pleural Effusion     → pos_weight = 0.50


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 8: ARQUITECTURA — ResNet-50 CheXpert + Correlación de Etiquetas + Metadatos
# ══════════════════════════════════════════════════════════════════════════════
#
# Diagrama de la arquitectura:
#
#   Imagen CXR (3×224×224)
#       │
#       ▼
#   ResNet-50 backbone (pesos CheXpert pre-entrenados)
#       │
#   GAP → embedding [B, 2048]
#       │
#       ├─────────────────────────────────────┐
#       │                                     │
#   Proyección imagen          Rama metadatos (si use_meta_branch=True)
#   [B, 2048] → [B, 512]       [B, 13] → [B, 64] → [B, 64]
#       │                                     │
#       └──────────── Concatenación ──────────┘
#                         │
#                    [B, 512+64] o [B, 512]
#                         │
#                    BatchNorm + Dropout
#                         │
#                   Head lineal → [B, N_LABELS]  (logits crudos)
#                         │
#   Módulo correlación (si use_label_correlation=True):
#   logits [B, N] → L_corr (NxN aprendible) → logits_corr [B, N]
#   logits_final = logits + logits_corr  (skip connection)
#                         │
#                    sigmoid → [B, N_LABELS] (probabilidades)
#
# ══════════════════════════════════════════════════════════════════════════════

META_DIM   = 13   # Dimensión del vector de metadatos (ver _encode_metadata)
META_EMBED = 64   # Dimensión del embedding de metadatos tras la MLP auxiliar
IMG_PROJ   = 512  # Dimensión de proyección del embedding de imagen


class LabelCorrelationModule(nn.Module):
    """
    Módulo de correlación entre etiquetas: capa lineal NxN aprendible.
    
    Cada logit de salida recibe información de los logits de las otras etiquetas
    antes de pasar por la sigmoid. Esto permite modelar explícitamente co-ocurrencias
    como Atelectasis–Derrame Pleural (Jaccard=0.30) o Cardiomegalia–Edema.
    
    Se implementa como una transformación lineal cuadrada sin sesgo (bias=False)
    para evitar que el sesgo absorba la información de correlación. Se añade
    una skip connection (logits originales + transformación) para estabilidad.
    """
    
    def __init__(self, n_labels: int = N_LABELS):
        super().__init__()
        self.n_labels = n_labels
        
        # Matriz de correlación aprendible: [N_LABELS × N_LABELS]
        # Inicialización con identidad escalada para comenzar cerca de "sin correlación"
        self.correlation = nn.Linear(n_labels, n_labels, bias=False)
        nn.init.eye_(self.correlation.weight)  # Inicialización identidad
        self.correlation.weight.data *= 0.1   # Escalar para comenzar pequeño
        
        # LayerNorm para estabilizar la señal de correlación
        self.norm = nn.LayerNorm(n_labels)
    
    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits: [B, N_LABELS]
        Returns:
            logits_corr: [B, N_LABELS] con información de co-ocurrencia añadida.
        """
        # Transformación lineal entre logits (información cruzada entre etiquetas)
        corr_signal = self.correlation(logits)
        
        # Skip connection: el modelo puede aprender a ignorar la correlación si no ayuda
        return self.norm(logits + corr_signal)


class MetadataBranch(nn.Module):
    """
    Rama auxiliar para procesar metadatos clínicos (edad, género, raza, etc.).
    
    MLP pequeña: meta_dim → 128 → meta_embed con BatchNorm y ReLU.
    """
    
    def __init__(self, meta_dim: int = META_DIM, meta_embed: int = META_EMBED):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(meta_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, meta_embed),
            nn.BatchNorm1d(meta_embed),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class CXRResNet50(nn.Module):
    """
    Modelo principal: ResNet-50 con pesos CheXpert + módulo de correlación + rama metadatos.
    
    Args:
        n_labels              : Número de etiquetas multilabel.
        dropout_rate          : Tasa de dropout en la cabeza clasificadora.
        use_label_correlation : Si True, añade el módulo 6×6 de co-ocurrencia.
        use_meta_branch       : Si True, añade la rama de metadatos clínicos.
        pretrained_source     : 'chexpert' (torchxrayvision) o 'imagenet' (torchvision).
    """
    
    def __init__(
        self,
        n_labels: int                = N_LABELS,
        dropout_rate: float          = 0.5,
        use_label_correlation: bool  = True,
        use_meta_branch: bool        = True,
        pretrained_source: str       = "chexpert",
    ):
        super().__init__()
        
        self.n_labels              = n_labels
        self.use_label_correlation = use_label_correlation
        self.use_meta_branch       = use_meta_branch
        
        # ── 1. Backbone ResNet-50 ─────────────────────────────────────────────
        if pretrained_source == "chexpert":
            # torchxrayvision: modelo DenseNet/ResNet entrenado sobre CheXpert + MIMIC.
            # Usamos el modelo "resnet50" disponible en la librería.
            # Los pesos son los más adecuados para nuestro dominio (CXR torácica).
            try:
                # Carga el modelo ResNet-50 pre-entrenado en CheXpert (14 patologías)
                xrv_model = xrv.models.ResNet(weights="resnet50-res512-all")
                self.backbone = xrv_model.model  # El backbone puro sin la cabeza
                backbone_out_dim = 2048
                print("   ✓ Backbone: ResNet-50 con pesos CheXpert (torchxrayvision)")
            except Exception as e:
                print(f"   ⚠ torchxrayvision falló ({e}), usando ResNet-50 ImageNet.")
                backbone = tv_models.resnet50(pretrained=True)
                self.backbone = nn.Sequential(*list(backbone.children())[:-2])
                backbone_out_dim = 2048
        else:
            # Fallback: ResNet-50 estándar con pesos ImageNet
            backbone = tv_models.resnet50(pretrained=True)
            self.backbone = nn.Sequential(*list(backbone.children())[:-2])
            backbone_out_dim = 2048
            print("   ✓ Backbone: ResNet-50 con pesos ImageNet (torchvision)")
        
        # Global Average Pooling para colapsar el mapa espacial a un vector
        self.gap = nn.AdaptiveAvgPool2d(1)
        
        # ── 2. Proyección del embedding de imagen ─────────────────────────────
        self.img_proj = nn.Sequential(
            nn.Linear(backbone_out_dim, IMG_PROJ),
            nn.BatchNorm1d(IMG_PROJ),
            nn.ReLU(inplace=True),
        )
        
        # ── 3. Rama de metadatos (opcional) ───────────────────────────────────
        if use_meta_branch:
            self.meta_branch = MetadataBranch(META_DIM, META_EMBED)
            fusion_dim = IMG_PROJ + META_EMBED
        else:
            self.meta_branch = None
            fusion_dim = IMG_PROJ
        
        # ── 4. Cabeza clasificadora ───────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(fusion_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.5),  # Segundo dropout más suave
            nn.Linear(256, n_labels),         # Logits crudos (sin sigmoid)
        )
        
        # ── 5. Módulo de correlación entre etiquetas (opcional) ───────────────
        if use_label_correlation:
            self.label_corr = LabelCorrelationModule(n_labels)
        else:
            self.label_corr = None
        
        # Inicialización de la cabeza con He normal (apropiado para ReLU)
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def freeze_backbone(self):
        """Congela todos los parámetros del backbone (fase inicial de entrenamiento)."""
        for param in self.backbone.parameters():
            param.requires_grad = False
        print("   🔒 Backbone congelado.")
    
    def unfreeze_layers(self, strategy: str = "layer4"):
        """
        Descongela capas del backbone según la estrategia especificada.
        
        Args:
            strategy: 'layer4' → solo último bloque residual.
                      'layer3_4' → dos últimos bloques.
                      'all' → todo el backbone.
        """
        # Primero, asegurarse de que todo está congelado
        for param in self.backbone.parameters():
            param.requires_grad = False
        
        if strategy == "all":
            for param in self.backbone.parameters():
                param.requires_grad = True
            print("   🔓 Backbone completamente descongelado.")
        elif strategy == "layer4":
            # Descongelar solo layer4 (último bloque residual)
            for name, param in self.backbone.named_parameters():
                if "layer4" in name:
                    param.requires_grad = True
            print("   🔓 Backbone descongelado: layer4.")
        elif strategy == "layer3_4":
            for name, param in self.backbone.named_parameters():
                if "layer3" in name or "layer4" in name:
                    param.requires_grad = True
            print("   🔓 Backbone descongelado: layer3 + layer4.")
    
    def forward(self, image: torch.Tensor, metadata: torch.Tensor = None) -> dict:
        """
        Forward pass completo.
        
        Args:
            image    : [B, 3, H, W] tensor de imagen normalizado.
            metadata : [B, 13] tensor de metadatos (None si use_meta_branch=False).
        
        Returns:
            dict con 'logits' [B, N_LABELS] y 'probs' [B, N_LABELS].
        """
        # ── Backbone → embedding ──────────────────────────────────────────────
        feat_map = self.backbone(image)         # [B, 2048, H', W']
        feat_vec = self.gap(feat_map)           # [B, 2048, 1, 1]
        feat_vec = feat_vec.flatten(1)          # [B, 2048]
        feat_proj = self.img_proj(feat_vec)     # [B, 512]
        
        # ── Metadatos (opcional) ──────────────────────────────────────────────
        if self.use_meta_branch and metadata is not None:
            meta_emb = self.meta_branch(metadata)    # [B, 64]
            fused = torch.cat([feat_proj, meta_emb], dim=1)  # [B, 576]
        else:
            fused = feat_proj                         # [B, 512]
        
        # ── Cabeza clasificadora ──────────────────────────────────────────────
        logits = self.head(fused)               # [B, N_LABELS]
        
        # ── Módulo de correlación entre etiquetas (opcional) ──────────────────
        if self.use_label_correlation:
            logits = self.label_corr(logits)    # [B, N_LABELS]
        
        # Probabilidades finales (aplicamos sigmoid en inferencia, no en pérdida)
        probs = torch.sigmoid(logits)
        
        return {"logits": logits, "probs": probs}


print("✅ Arquitectura CXRResNet50 definida.")
print("   Diagrama: ResNet-50 (CheXpert) → GAP → Proyección → [Meta?] → Cabeza → [Correlación?] → Sigmoid")


✅ Arquitectura CXRResNet50 definida.
   Diagrama: ResNet-50 (CheXpert) → GAP → Proyección → [Meta?] → Cabeza → [Correlación?] → Sigmoid


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 9: UTILIDADES — MÉTRICAS, UMBRALES ÓPTIMOS, OPTIMIZADOR
# ══════════════════════════════════════════════════════════════════════════════
#
# FIXES aplicados:
#   [FIX 1] ReduceLROnPlateau: eliminado verbose=False, deprecado en PyTorch ≥2.2
#            y eliminado definitivamente en ≥2.4. Su presencia lanza TypeError
#            en __init__ impidiendo cualquier entrenamiento con scheduler=plateau.
#   [FIX 2] build_optimizer_and_scheduler acepta ahora un parámetro opcional
#            `num_epochs_override` para que train_model pueda reconstruir el
#            optimizador al descongelar el backbone pasando solo las épocas
#            RESTANTES, evitando que OneCycleLR exceda el total de pasos y
#            lance ValueError en el segundo ciclo de entrenamiento.
# ══════════════════════════════════════════════════════════════════════════════


def compute_multilabel_metrics(
    all_probs:  np.ndarray,         # [N, N_LABELS] probabilidades predichas
    all_labels: np.ndarray,         # [N, N_LABELS] etiquetas verdaderas (0/1)
    all_masks:  np.ndarray,         # [N, N_LABELS] máscaras de observabilidad
    thresholds: np.ndarray = None,  # [N_LABELS] umbrales de clasificación (default=0.5)
    labels: list = LABELS,
) -> dict:
    """
    Calcula métricas multilabel por etiqueta y macro-promedio.
    Solo considera pares (muestra, etiqueta) donde mask=1.

    Returns:
        dict con AUC-ROC, AP, F1 por etiqueta y macro-promedio.
    """
    if thresholds is None:
        thresholds = np.full(len(labels), 0.5)

    metrics = {}
    auc_list, ap_list, f1_list = [], [], []

    for i, lbl in enumerate(labels):
        # Solo usar pares con etiqueta observada
        valid_mask = all_masks[:, i] == 1
        y_true = all_labels[valid_mask, i]
        y_prob = all_probs[valid_mask, i]
        y_pred = (y_prob >= thresholds[i]).astype(float)

        n_pos = y_true.sum()
        n_neg = (1 - y_true).sum()

        if n_pos < 2 or n_neg < 2:
            # AUC indefinida con una sola clase presente en el subset
            auc = float("nan")
            ap  = float("nan")
        else:
            auc = roc_auc_score(y_true, y_prob)
            ap  = average_precision_score(y_true, y_prob)

        f1 = f1_score(y_true, y_pred, zero_division=0)

        metrics[lbl] = {
            "AUC"  : auc,
            "AP"   : ap,
            "F1"   : f1,
            "n_pos": int(n_pos),
            "n_neg": int(n_neg),
        }

        if not np.isnan(auc):
            auc_list.append(auc)
            ap_list.append(ap)
        f1_list.append(f1)

    metrics["macro_AUC"] = float(np.nanmean(auc_list)) if auc_list else float("nan")
    metrics["macro_AP"]  = float(np.nanmean(ap_list))  if ap_list  else float("nan")
    metrics["macro_F1"]  = float(np.nanmean(f1_list))  if f1_list  else float("nan")

    return metrics


def find_optimal_thresholds(
    all_probs:  np.ndarray,
    all_labels: np.ndarray,
    all_masks:  np.ndarray,
    labels: list = LABELS,
    n_thresholds: int = 50,
) -> np.ndarray:
    """
    Busca el umbral óptimo por etiqueta que maximiza el F1-score.
    Se aplica sobre el conjunto de validación interno del fold.

    Returns:
        Array [N_LABELS] con umbrales óptimos en [0.1, 0.9].
    """
    thresholds = np.full(len(labels), 0.5)

    for i, lbl in enumerate(labels):
        valid_mask = all_masks[:, i] == 1
        y_true = all_labels[valid_mask, i]
        y_prob = all_probs[valid_mask, i]

        if y_true.sum() < 2:
            continue  # No suficientes positivos para optimizar el umbral

        best_f1  = -1.0
        best_thr =  0.5

        for thr in np.linspace(0.1, 0.9, n_thresholds):
            y_pred = (y_prob >= thr).astype(float)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            if f1 > best_f1:
                best_f1  = f1
                best_thr = thr

        thresholds[i] = best_thr

    return thresholds


def build_optimizer_and_scheduler(
    model,
    config: dict,
    train_loader_len: int,
    num_epochs_override: int = None,   # [FIX 2] épocas a usar (None → config["num_epochs"])
):
    """
    Construye el optimizador AdamW con LR diferenciada por grupo de parámetros
    (backbone más lento que la cabeza) y el scheduler correspondiente.

    Args:
        model               : Instancia de CXRResNet50.
        config              : Diccionario de hiperparámetros del fold actual.
        train_loader_len    : Número de batches por época (necesario para OneCycleLR).
        num_epochs_override : Si se indica, sustituye a config["num_epochs"].
                              Úsalo al reconstruir el optimizador tras el descongelado
                              del backbone para pasar solo las épocas restantes y que
                              OneCycleLR no exceda el total de pasos planificados.

    Returns:
        (optimizer, scheduler)  — scheduler puede ser None si config["scheduler"]
        no coincide con ninguno de los tres valores conocidos.
    """
    # [FIX 2] épocas efectivas para el scheduler
    num_epochs = num_epochs_override if num_epochs_override is not None else config["num_epochs"]

    # ── Grupos de parámetros con LR diferenciada ──────────────────────────────
    # El backbone usa una LR 10× menor para no degradar los pesos CheXpert.
    # La cabeza (proyección, head, correlación, metadatos) puede aprender más rápido.
    head_params = (
        list(model.img_proj.parameters())
        + list(model.head.parameters())
        + (list(model.label_corr.parameters())  if model.label_corr  else [])
        + (list(model.meta_branch.parameters()) if model.meta_branch else [])
    )

    param_groups = [
        {"params": model.backbone.parameters(), "lr": config["lr_backbone"], "name": "backbone"},
        {"params": head_params,                 "lr": config["lr_head"],     "name": "head"},
    ]

    optimizer = torch.optim.AdamW(param_groups, weight_decay=config["weight_decay"])

    # ── Scheduler ─────────────────────────────────────────────────────────────
    sched_name = config["scheduler"]

    if sched_name == "cosine":
        # Decaimiento coseno suave desde LR inicial hasta eta_min a lo largo
        # de todas las épocas. Comportamiento estable y predecible.
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=num_epochs,
            eta_min=1e-7,
        )

    elif sched_name == "onecycle":
        # Warm-up (10% de épocas) + annealing coseno en una sola pasada.
        # Muy eficiente con datasets médicos de tamaño moderado.
        # max_lr = 10× la LR base de cada grupo (regla empírica estándar).
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=[config["lr_backbone"] * 10, config["lr_head"] * 10],
            steps_per_epoch=train_loader_len,
            epochs=num_epochs,
            pct_start=0.1,
        )

    elif sched_name == "plateau":
        # [FIX 1] verbose=False eliminado: deprecado en PyTorch ≥2.2,
        # removido en ≥2.4 → lanzaba TypeError en __init__.
        # ReduceLROnPlateau reduce la LR a la mitad si val_loss no mejora
        # durante 3 épocas consecutivas. Conservador pero robusto.
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=3,
        )

    else:
        scheduler = None

    return optimizer, scheduler


print("✅ Utilidades definidas: compute_multilabel_metrics | find_optimal_thresholds | build_optimizer_and_scheduler")
print("   [FIX 1] ReduceLROnPlateau: verbose=False eliminado (PyTorch ≥2.2/2.4 compatible)")
print("   [FIX 2] build_optimizer_and_scheduler acepta num_epochs_override para reconstrucción post-unfreeze")

✅ Utilidades definidas: compute_multilabel_metrics | find_optimal_thresholds | build_optimizer_and_scheduler
   [FIX 1] ReduceLROnPlateau: verbose=False eliminado (PyTorch ≥2.2/2.4 compatible)
   [FIX 2] build_optimizer_and_scheduler acepta num_epochs_override para reconstrucción post-unfreeze


In [10]:
# =============================================================================
# CELDA 10 — FUNCIONES DE ENTRENAMIENTO Y EVALUACIÓN POR ÉPOCA (CON TQDM + NPY)
# =============================================================================
 
def train_one_epoch(
    model, loader, optimizer, scheduler, criterion, config: dict
) -> float:
    """
    Entrena el modelo durante una época completa.
 
    Returns:
        Pérdida media de entrenamiento en la época.
    """
    model.train()
    total_loss = 0.0
    n_batches  = 0
 
    pbar = tqdm(loader, desc="  Train", leave=False, unit="batch",
                bar_format="{l_bar}{bar:25}{r_bar}")
    for batch in pbar:
        images   = batch["image"].to(DEVICE)
        labels   = batch["labels"].to(DEVICE)
        masks    = batch["mask"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE) if config.get("use_meta_branch") else None
 
        optimizer.zero_grad()
 
        output = model(images, metadata)
        loss   = criterion(output["logits"], labels, masks)
 
        loss.backward()
 
        # Gradient clipping: evita explosión del gradiente, especialmente
        # durante las primeras épocas con backbone descongelado.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
 
        optimizer.step()
 
        # OneCycleLR necesita step por batch (no por época)
        if config.get("scheduler") == "onecycle" and scheduler is not None:
            scheduler.step()
 
        total_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")
 
    return total_loss / max(n_batches, 1)
 
 
@torch.no_grad()
def evaluate(model, loader, criterion, config: dict) -> tuple:
    """
    Evalúa el modelo en un DataLoader (val o test) sin gradientes.
 
    Returns:
        (val_loss, all_probs, all_labels, all_masks)
    """
    model.eval()
    total_loss = 0.0
    n_batches  = 0
 
    probs_list, labels_list, masks_list = [], [], []
 
    for batch in tqdm(loader, desc="  Eval ", leave=False, unit="batch",
                      bar_format="{l_bar}{bar:25}{r_bar}"):
        images   = batch["image"].to(DEVICE)
        labels   = batch["labels"].to(DEVICE)
        masks    = batch["mask"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE) if config.get("use_meta_branch") else None
 
        output = model(images, metadata)
        loss   = criterion(output["logits"], labels, masks)
 
        total_loss += loss.item()
        n_batches  += 1
 
        probs_list.append(output["probs"].cpu().numpy())
        labels_list.append(labels.cpu().numpy())
        masks_list.append(masks.cpu().numpy())
 
    all_probs  = np.concatenate(probs_list,  axis=0)
    all_labels = np.concatenate(labels_list, axis=0)
    all_masks  = np.concatenate(masks_list,  axis=0)
 
    return total_loss / max(n_batches, 1), all_probs, all_labels, all_masks
 
 
def train_model(
    model, df_train_fold, df_val_fold, config: dict, verbose: bool = True
) -> tuple:
    """
    Entrena un modelo completo dado un config de hiperparámetros y los DataFrames
    de train y validación del fold actual.
 
    Gestiona:
      - Pipeline de augmentación según config['augmentation_level'].
      - Congelado/descongelado progresivo del backbone.
      - Early stopping basado en val_loss (paciencia=5 épocas).
      - Guardado del mejor modelo del fold.
      - Búsqueda de umbrales óptimos en el val set.
 
    Returns:
        (best_model_state_dict, best_val_metrics, best_thresholds, history)
    """
    # ── Datasets y loaders ────────────────────────────────────────────────────
    aug_train = get_augmentation_pipeline(config["augmentation_level"])
    aug_val   = get_augmentation_pipeline("test")  # Sin augmentación en val
 
    # cxr_npy se pasa como atributo temporal del DataFrame desde el Nested CV
    cxr_npy_tr = getattr(df_train_fold, "_cxr_npy", None)
    cxr_npy_vl = getattr(df_val_fold,   "_cxr_npy", None)
 
    if cxr_npy_tr is None or cxr_npy_vl is None:
        # Fallback para pruebas rapidas fuera del Nested CV
        cxr_npy_tr = cxr_npy_train[:len(df_train_fold)]
        cxr_npy_vl = cxr_npy_train[:len(df_val_fold)]
 
    ds_train = CXRMultilabelDataset(
        df_train_fold, cxr_npy_tr, aug_train,
        uncertainty_policy=config["uncertainty_policy"]
    )
    ds_val = CXRMultilabelDataset(
        df_val_fold, cxr_npy_vl, aug_val,
        uncertainty_policy=config["uncertainty_policy"]
    )
 
    loader_train = DataLoader(
        ds_train, batch_size=config["batch_size"],
        shuffle=True, num_workers=4, pin_memory=True, drop_last=True
    )
    loader_val = DataLoader(
        ds_val, batch_size=config["batch_size"] * 2,
        shuffle=False, num_workers=4, pin_memory=True
    )
 
    # ── Optimizador y scheduler ───────────────────────────────────────────────
    optimizer, scheduler = build_optimizer_and_scheduler(model, config, len(loader_train))
 
    # ── Congelado inicial del backbone ────────────────────────────────────────
    model.freeze_backbone()
 
    # ── Early stopping ────────────────────────────────────────────────────────
    best_val_loss      = float("inf")
    best_model_state   = None
    best_val_metrics   = None
    best_thresholds    = np.full(N_LABELS, 0.5)
    patience_counter   = 0
    PATIENCE           = 5
 
    history = {"train_loss": [], "val_loss": [], "val_auc_macro": []}
 
    epoch_iter = tqdm(
        range(config["num_epochs"]),
        desc="  Epocas",
        unit="epoca",
        bar_format="{l_bar}{bar:30}{r_bar}",
        disable=not verbose,
    )
    for epoch in epoch_iter:
 
        # ── Estrategia de descongelado progresivo ─────────────────────────────
        if epoch == config["unfreeze_epoch"]:
            model.unfreeze_layers(config["unfreeze_layers"])
            optimizer, scheduler = build_optimizer_and_scheduler(model, config, len(loader_train))
 
        # ── Entrenamiento ─────────────────────────────────────────────────────
        train_loss = train_one_epoch(model, loader_train, optimizer, scheduler, criterion, config)
 
        # ── Evaluación en validación ──────────────────────────────────────────
        val_loss, all_probs, all_labels, all_masks = evaluate(model, loader_val, criterion, config)
 
        # Métricas con umbral 0.5 (para monitoreo durante entrenamiento)
        val_metrics = compute_multilabel_metrics(all_probs, all_labels, all_masks)
 
        # Scheduler paso (excepto OneCycleLR que ya hizo step por batch)
        if scheduler is not None and config["scheduler"] != "onecycle":
            if config["scheduler"] == "plateau":
                scheduler.step(val_loss)
            else:
                scheduler.step()
 
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_auc_macro"].append(val_metrics["macro_AUC"])
 
        if verbose:
            epoch_iter.set_postfix(
                tr_loss=f"{train_loss:.4f}",
                val_loss=f"{val_loss:.4f}",
                val_AUC=f"{val_metrics['macro_AUC']:.4f}",
            )
 
        # ── Early stopping ────────────────────────────────────────────────────
        if val_loss < best_val_loss - 1e-4:
            best_val_loss    = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
 
            if config.get("threshold_search"):
                best_thresholds = find_optimal_thresholds(all_probs, all_labels, all_masks)
 
            best_val_metrics = compute_multilabel_metrics(
                all_probs, all_labels, all_masks, thresholds=best_thresholds
            )
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                if verbose:
                    tqdm.write(f"     Early stopping en epoca {epoch+1}.")
                break
 
    return best_model_state, best_val_metrics, best_thresholds, history
 
 
print("✅ Funciones de entrenamiento y evaluación definidas.")

✅ Funciones de entrenamiento y evaluación definidas.


In [11]:
# =============================================================================
# CELDA 11 — NESTED CROSS-VALIDATION (CON TQDM + PROPAGACIÓN NPY)
# =============================================================================
 
import gc  # Para gc.collect() en liberación de memoria (FIX 4)
 
K_OUTER = 5
K_INNER = 3
 
df_cv     = df_train.copy().reset_index(drop=True)
n_samples = len(df_cv)
 
print("═" * 70)
print("  NESTED CROSS-VALIDATION — ResNet-50 CXR Multilabel")
print("═" * 70)
print(f"  Muestras en CV : {n_samples:,}")
print(f"  K externo      : {K_OUTER} folds  (estimación del rendimiento real)")
print(f"  K interno      : {K_INNER} folds  (selección de hiperparámetros)")
print(f"  Configs en RS  : {len(SAMPLED_CONFIGS)} configuraciones muestreadas")
print(f"  Total runs     : {K_OUTER} × {len(SAMPLED_CONFIGS)} × {K_INNER} = "
      f"{K_OUTER * len(SAMPLED_CONFIGS) * K_INNER} entrenamientos internos "
      f"+ {K_OUTER} reentrenamientos finales")
print("═" * 70)
 
outer_results = []
 
outer_kf = KFold(n_splits=K_OUTER, shuffle=True, random_state=SEED)
 
for outer_fold_idx, (outer_train_idx, outer_test_idx) in enumerate(
    outer_kf.split(np.arange(n_samples))
):
    print(f"\n{'─' * 70}")
    print(f"  FOLD EXTERNO {outer_fold_idx + 1}/{K_OUTER}")
    print(f"  Train outer: {len(outer_train_idx):,} | Test outer: {len(outer_test_idx):,}")
    print(f"{'─' * 70}")
 
    df_outer_train = df_cv.iloc[outer_train_idx].reset_index(drop=True)
    df_outer_test  = df_cv.iloc[outer_test_idx].reset_index(drop=True)
 
    # ── LOOP INTERNO: búsqueda de hiperparámetros ─────────────────────────────
    best_inner_auc = -1.0
    best_config    = SAMPLED_CONFIGS[0]
 
    inner_kf = KFold(n_splits=K_INNER, shuffle=True, random_state=SEED)
 
    config_pbar = tqdm(
        enumerate(SAMPLED_CONFIGS), total=len(SAMPLED_CONFIGS),
        desc=f"  [Fold ext {outer_fold_idx+1}/{K_OUTER}] Configs",
        unit="config",
        bar_format="{l_bar}{bar:30}{r_bar}",
    )
    for config_idx, config in config_pbar:
 
        config_auc_scores = []
 
        for inner_fold_idx, (inner_train_idx, inner_val_idx) in enumerate(
            tqdm(
                inner_kf.split(np.arange(len(df_outer_train))),
                total=K_INNER,
                desc=f"    Inner folds (config {config_idx+1})",
                unit="fold",
                leave=False,
                bar_format="{l_bar}{bar:20}{r_bar}",
            )
        ):
            df_inner_train = df_outer_train.iloc[inner_train_idx].reset_index(drop=True)
            df_inner_val   = df_outer_train.iloc[inner_val_idx].reset_index(drop=True)
 
            model = CXRResNet50(
                n_labels=N_LABELS,
                dropout_rate=config["dropout_rate"],
                use_label_correlation=config["use_label_correlation"],
                use_meta_branch=config["use_meta_branch"],
                pretrained_source="chexpert",
            ).to(DEVICE)
 
            # Propagar sub-arrays npy como atributos temporales del DataFrame
            df_inner_train._cxr_npy = cxr_npy_train[outer_train_idx[inner_train_idx]]
            df_inner_val._cxr_npy   = cxr_npy_train[outer_train_idx[inner_val_idx]]
 
            # Entrenar con la config actual.
            _, val_metrics, _, _ = train_model(
                model, df_inner_train, df_inner_val, config, verbose=False
            )
 
            # [FIX 3] Guardia NaN
            auc = (
                val_metrics.get("macro_AUC", float("nan"))
                if val_metrics is not None
                else float("nan")
            )
            config_auc_scores.append(auc)
 
            # [FIX 4] Liberación de memoria robusta
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
 
        mean_inner_auc = float(np.nanmean(config_auc_scores))
 
        config_pbar.set_postfix(AUC=f"{mean_inner_auc:.4f}", best=f"{best_inner_auc:.4f}")
        tqdm.write(
            f"    Config {config_idx + 1:2d}/{len(SAMPLED_CONFIGS)} | "
            f"mean_inner_AUC={mean_inner_auc:.4f} | "
            f"corr={config['use_label_correlation']} | "
            f"meta={config['use_meta_branch']} | "
            f"lr_head={config['lr_head']} | "
            f"policy={config['uncertainty_policy']} | "
            f"aug={config['augmentation_level']}"
        )
 
        if mean_inner_auc > best_inner_auc:
            best_inner_auc = mean_inner_auc
            best_config    = config
 
    print(f"\n  ✅ Mejor config fold externo {outer_fold_idx + 1}: "
          f"AUC_inner={best_inner_auc:.4f}")
    for k, v in best_config.items():
        print(f"     {k:30s} = {v}")
 
    # ── Reentrenamiento con la mejor config sobre todo train_outer ────────────
    print(f"\n  🔁 Reentrenando con mejor config sobre train_outer completo...")
 
    model_final = CXRResNet50(
        n_labels=N_LABELS,
        dropout_rate=best_config["dropout_rate"],
        use_label_correlation=best_config["use_label_correlation"],
        use_meta_branch=best_config["use_meta_branch"],
        pretrained_source="chexpert",
    ).to(DEVICE)
 
    # Propagar arrays npy al reentrenamiento final del fold externo
    df_outer_train._cxr_npy = cxr_npy_train[outer_train_idx]
    df_outer_test._cxr_npy  = cxr_npy_train[outer_test_idx]
 
    best_state, _, best_thresholds, history = train_model(
        model_final, df_outer_train, df_outer_test, best_config, verbose=True
    )
 
    # ── Evaluación en test externo con el mejor checkpoint ────────────────────
    model_final.load_state_dict(best_state)
 
    aug_test  = get_augmentation_pipeline("test")
    cxr_npy_outer_test = cxr_npy_train[outer_test_idx]
    ds_test_o = CXRMultilabelDataset(
        df_outer_test, cxr_npy_outer_test, aug_test,
        uncertainty_policy=best_config["uncertainty_policy"],
    )
    loader_test_o = DataLoader(
        ds_test_o,
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )
 
    _, test_probs, test_labels_arr, test_masks = evaluate(
        model_final, loader_test_o, criterion, best_config
    )
    test_metrics = compute_multilabel_metrics(
        test_probs, test_labels_arr, test_masks, thresholds=best_thresholds
    )
 
    print(f"\n  📊 Métricas test_outer fold {outer_fold_idx + 1}:")
    print(f"     macro_AUC = {test_metrics['macro_AUC']:.4f}")
    print(f"     macro_AP  = {test_metrics['macro_AP']:.4f}")
    print(f"     macro_F1  = {test_metrics['macro_F1']:.4f}")
    print(f"     {'Etiqueta':22s} | {'AUC':>6} | {'F1':>6} | {'N+':>5} | {'N-':>5}")
    print(f"     {'─' * 52}")
    for lbl in LABELS:
        m       = test_metrics[lbl]
        auc_str = f"{m['AUC']:.4f}" if not np.isnan(m["AUC"]) else "  N/A"
        print(f"     {lbl:22s} | {auc_str:>6} | {m['F1']:>6.4f} | "
              f"{m['n_pos']:>5} | {m['n_neg']:>5}")
 
    # ── Guardar checkpoint del fold externo ───────────────────────────────────
    fold_checkpoint_path = OUTPUT_DIR / f"model_outer_fold{outer_fold_idx + 1}.pt"
    torch.save(
        {
            "model_state_dict": best_state,
            "best_config"     : best_config,
            "thresholds"      : best_thresholds.tolist(),
            "test_metrics"    : test_metrics,
            "outer_fold"      : outer_fold_idx + 1,
        },
        fold_checkpoint_path,
    )
    print(f"\n  💾 Checkpoint guardado: {fold_checkpoint_path}")
 
    outer_results.append(
        {
            "outer_fold"    : outer_fold_idx + 1,
            "best_config"   : best_config,
            "best_inner_auc": best_inner_auc,
            "test_metrics"  : test_metrics,
            "thresholds"    : best_thresholds.tolist(),
            "history"       : history,
            "checkpoint"    : str(fold_checkpoint_path),
        }
    )
 
    # [FIX 4] Liberación de memoria robusta tras cada fold externo
    del model_final
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
 
print("\n" + "═" * 70)
print("  NESTED CV COMPLETADO")
print("═" * 70)
auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]
f1_macros  = [r["test_metrics"]["macro_F1"]  for r in outer_results]
print(f"  AUC macro por fold : {[f'{a:.4f}' for a in auc_macros]}")
print(f"  Media AUC macro    : {np.mean(auc_macros):.4f} ± {np.std(auc_macros):.4f}")
print(f"  Media F1  macro    : {np.mean(f1_macros):.4f}  ± {np.std(f1_macros):.4f}")
print("═" * 70)

══════════════════════════════════════════════════════════════════════
  NESTED CROSS-VALIDATION — ResNet-50 CXR Multilabel
══════════════════════════════════════════════════════════════════════
  Muestras en CV : 10,000
  K externo      : 5 folds  (estimación del rendimiento real)
  K interno      : 3 folds  (selección de hiperparámetros)
  Configs en RS  : 20 configuraciones muestreadas
  Total runs     : 5 × 20 × 3 = 300 entrenamientos internos + 5 reentrenamientos finales
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
  FOLD EXTERNO 1/5
  Train outer: 8,000 | Test outer: 2,000
──────────────────────────────────────────────────────────────────────


  [Fold ext 1/5] Configs:   0%|                              | 0/20 [00:01<?, ?config/s]

   ✓ Backbone: ResNet-50 con pesos CheXpert (torchxrayvision)


NameError: name 'cxr_npy_train' is not defined

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 12: RESUMEN DE RESULTADOS DEL NESTED CV
# ══════════════════════════════════════════════════════════════════════════════
# Esta es la estimación OFICIAL del rendimiento del modelo a reportar en el TFM.
# Corresponde a la media y desviación estándar del AUC macro sobre los K_OUTER
# folds externos, que nunca participaron en la selección de hiperparámetros.
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("  RESULTADOS NESTED CV — ResNet-50 CXR Multilabel")
print("  (Estimación no sesgada del rendimiento generalizable)")
print("═" * 70)

# AUC macro por fold externo
auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]
f1_macros  = [r["test_metrics"]["macro_F1"]  for r in outer_results]

print(f"\n  AUC macro por fold externo: {[f'{a:.4f}' for a in auc_macros]}")
print(f"  → Media AUC macro : {np.mean(auc_macros):.4f} ± {np.std(auc_macros):.4f}")
print(f"  → Media F1  macro : {np.mean(f1_macros):.4f} ± {np.std(f1_macros):.4f}")

# AUC por etiqueta promediado sobre folds externos
print("\n  AUC por etiqueta (media ± std sobre folds externos):")
for lbl in LABELS:
    aucs = [r["test_metrics"][lbl]["AUC"] for r in outer_results]
    print(f"    {lbl:22s}: {np.nanmean(aucs):.4f} ± {np.nanstd(aucs):.4f}")

# Configuraciones ganadoras por fold externo
print("\n  Configuraciones ganadoras por fold externo:")
for r in outer_results:
    cfg = r["best_config"]
    print(f"    Fold {r['outer_fold']}: inner_AUC={r['best_inner_auc']:.4f} | "
          f"corr={cfg['use_label_correlation']} | meta={cfg['use_meta_branch']} | "
          f"policy={cfg['uncertainty_policy']} | aug={cfg['augmentation_level']}")

# Guardar resumen completo en JSON
summary = {
    "mean_macro_AUC" : float(np.mean(auc_macros)),
    "std_macro_AUC"  : float(np.std(auc_macros)),
    "mean_macro_F1"  : float(np.mean(f1_macros)),
    "std_macro_F1"   : float(np.std(f1_macros)),
    "fold_results"   : [
        {
            "outer_fold"    : r["outer_fold"],
            "macro_AUC"     : r["test_metrics"]["macro_AUC"],
            "macro_F1"      : r["test_metrics"]["macro_F1"],
            "best_config"   : r["best_config"],
            "thresholds"    : r["thresholds"],
            "checkpoint"    : r["checkpoint"],
            "per_label"     : {lbl: r["test_metrics"][lbl] for lbl in LABELS},
        }
        for r in outer_results
    ],
}

summary_path = OUTPUT_DIR / "nested_cv_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\n  💾 Resumen guardado en: {summary_path}")
print("═" * 70)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 13: VISUALIZACIÓN — CURVAS DE ENTRENAMIENTO Y RESULTADOS DEL CV
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("ResNet-50 CXR — Nested CV Results", fontsize=14, fontweight="bold")

# ── Plot 1: Loss de entrenamiento y validación por fold externo ───────────────
ax = axes[0]
for r in outer_results:
    hist = r["history"]
    ax.plot(hist["train_loss"], label=f"Fold {r['outer_fold']} train", linestyle="--", alpha=0.6)
    ax.plot(hist["val_loss"],   label=f"Fold {r['outer_fold']} val",   alpha=0.9)
ax.set_xlabel("Época")
ax.set_ylabel("Masked BCE Loss")
ax.set_title("Curvas de Pérdida por Fold Externo")
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

# ── Plot 2: AUC macro en validación durante entrenamiento ────────────────────
ax = axes[1]
for r in outer_results:
    hist = r["history"]
    ax.plot(hist["val_auc_macro"], label=f"Fold {r['outer_fold']}", alpha=0.9)
ax.set_xlabel("Época")
ax.set_ylabel("AUC macro (val)")
ax.set_title("AUC Macro en Validación por Fold Externo")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.axhline(y=np.mean(auc_macros), color="red", linestyle=":", label="Media final")

# ── Plot 3: AUC por etiqueta (barras con error) ───────────────────────────────
ax = axes[2]
auc_by_label_mean = []
auc_by_label_std  = []
for lbl in LABELS:
    aucs = [r["test_metrics"][lbl]["AUC"] for r in outer_results]
    auc_by_label_mean.append(np.nanmean(aucs))
    auc_by_label_std.append(np.nanstd(aucs))

x_pos = np.arange(len(LABELS))
bars  = ax.bar(x_pos, auc_by_label_mean, yerr=auc_by_label_std,
               color="steelblue", alpha=0.7, capsize=4)
ax.set_xticks(x_pos)
ax.set_xticklabels([l[:10] for l in LABELS], rotation=35, ha="right", fontsize=8)
ax.set_ylabel("AUC-ROC (media ± std)")
ax.set_title("AUC por Etiqueta (Nested CV)")
ax.set_ylim([0, 1.05])
ax.axhline(y=0.5, color="gray", linestyle=":", alpha=0.5, label="Azar")
ax.axhline(y=np.mean(auc_macros), color="red", linestyle="--", alpha=0.7, label="Macro media")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

# Añadir valores encima de las barras
for bar, val in zip(bars, auc_by_label_mean):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nested_cv_results.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Figura guardada en: {OUTPUT_DIR / 'nested_cv_results.png'}")


In [ ]:
# =============================================================================
# CELDA 14 — EVALUACIÓN FINAL EN EL TEST SET OFICIAL (VERSIÓN NPY)
# =============================================================================
# Esta celda es la ÚNICA vez que se usa df_test (test_clean.csv).
# Se usa el modelo del fold externo con el AUC macro más alto.
#
# ⚠️ IMPORTANTE: Esta evaluación final se ejecuta UNA SOLA VEZ.
# =============================================================================
 
print("\n" + "═" * 70)
print("  EVALUACIÓN FINAL EN TEST SET OFICIAL (test_clean.csv)")
print("  ⚠ Esta evaluación solo se ejecuta UNA VEZ.")
print("═" * 70)
 
# Seleccionar el fold externo con mejor AUC como modelo representativo
best_outer_fold = outer_results[np.argmax(auc_macros)]
print(f"\n  Fold externo seleccionado: {best_outer_fold['outer_fold']} "
      f"(AUC_outer={auc_macros[np.argmax(auc_macros)]:.4f})")
 
# Cargar el checkpoint del mejor fold
checkpoint = torch.load(best_outer_fold["checkpoint"], map_location=DEVICE)
best_cfg   = checkpoint["best_config"]
 
model_test = CXRResNet50(
    n_labels=N_LABELS,
    dropout_rate=best_cfg["dropout_rate"],
    use_label_correlation=best_cfg["use_label_correlation"],
    use_meta_branch=best_cfg["use_meta_branch"],
    pretrained_source="chexpert",
).to(DEVICE)
 
model_test.load_state_dict(checkpoint["model_state_dict"])
best_thresholds_test = np.array(checkpoint["thresholds"])
 
# Dataset y loader de test oficial — usa cxr_npy_test
aug_test  = get_augmentation_pipeline("test")
ds_test   = CXRMultilabelDataset(
    df_test, cxr_npy_test, aug_test,
    uncertainty_policy=best_cfg["uncertainty_policy"]
)
loader_test = DataLoader(ds_test, batch_size=32, shuffle=False, num_workers=4)
 
# Evaluación
_, test_probs_final, test_labels_final, test_masks_final = evaluate(
    model_test, loader_test, criterion, best_cfg
)
 
# Métricas con umbrales óptimos
test_metrics_final = compute_multilabel_metrics(
    test_probs_final, test_labels_final, test_masks_final,
    thresholds=best_thresholds_test
)
 
print("\n  📊 MÉTRICAS FINALES EN TEST SET OFICIAL:")
print(f"  {'Etiqueta':22s} | {'AUC':>6} | {'AP':>6} | {'F1':>6} | {'N+':>5} | {'N-':>5}")
print("  " + "─" * 60)
for lbl in LABELS:
    m = test_metrics_final[lbl]
    auc_str = f"{m['AUC']:.4f}" if not np.isnan(m['AUC']) else "  N/A "
    ap_str  = f"{m['AP']:.4f}"  if not np.isnan(m['AP'])  else "  N/A "
    print(f"  {lbl:22s} | {auc_str:>6} | {ap_str:>6} | {m['F1']:>6.4f} | {m['n_pos']:>5} | {m['n_neg']:>5}")
 
print("  " + "─" * 60)
print(f"  {'MACRO':22s} | {test_metrics_final['macro_AUC']:>6.4f} | "
      f"{test_metrics_final['macro_AP']:>6.4f} | {test_metrics_final['macro_F1']:>6.4f}")
 
# Guardar métricas finales
final_results = {
    "test_set"          : "test_clean.csv",
    "model_fold_used"   : best_outer_fold["outer_fold"],
    "macro_AUC"         : test_metrics_final["macro_AUC"],
    "macro_AP"          : test_metrics_final["macro_AP"],
    "macro_F1"          : test_metrics_final["macro_F1"],
    "per_label"         : {lbl: test_metrics_final[lbl] for lbl in LABELS},
    "thresholds_used"   : best_thresholds_test.tolist(),
    "best_config"       : best_cfg,
}
 
with open(OUTPUT_DIR / "final_test_results.json", "w") as f:
    json.dump(final_results, f, indent=2, default=str)
 
print(f"\n  💾 Resultados finales guardados en: {OUTPUT_DIR / 'final_test_results.json'}")
print("═" * 70)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 15: ANÁLISIS DE ERRORES POR SUBGRUPO (EQUIDAD DEL MODELO)
# ══════════════════════════════════════════════════════════════════════════════
# Evalúa si el modelo tiene rendimiento diferenciado por subgrupos demográficos.
# Esto es especialmente relevante dado que el EDA mostró:
#   - Asociaciones género-etiqueta significativas (V de Cramér ≤ 0.051)
#   - Diferencias en prevalencia de Edema por género (56.5% F vs 51.4% M)
#   - Distribución racial heterogénea (dominada por WHITE y UNKNOWN)
# ══════════════════════════════════════════════════════════════════════════════

# Recuperar metadatos del test set para estratificación
# (las probabilidades se almacenaron en el mismo orden que df_test)
df_test_reset = df_test.reset_index(drop=True)

# Subgrupo: género
print("\n── AUC macro por GÉNERO ──────────────────────────────────────────────")
for gender_val, gender_name in [(0, "Femenino (0)"), (1, "Masculino (1)")]:
    mask_gender = df_test_reset["gender"] == gender_val
    idx_gender  = mask_gender[mask_gender].index.values
    
    if len(idx_gender) < 10:
        continue
    
    g_probs  = test_probs_final[idx_gender]
    g_labels = test_labels_final[idx_gender]
    g_masks  = test_masks_final[idx_gender]
    
    g_metrics = compute_multilabel_metrics(g_probs, g_labels, g_masks, thresholds=best_thresholds_test)
    print(f"  {gender_name:20s} (n={len(idx_gender):,}): macro_AUC={g_metrics['macro_AUC']:.4f} | macro_F1={g_metrics['macro_F1']:.4f}")

# Subgrupo: vista CXR (AP vs PA)
print("\n── AUC macro por VISTA CXR ───────────────────────────────────────────")
for view_val in ["AP", "PA"]:
    mask_view = df_test_reset["cxr_view"] == view_val
    idx_view  = mask_view[mask_view].index.values
    
    if len(idx_view) < 10:
        continue
    
    v_probs  = test_probs_final[idx_view]
    v_labels = test_labels_final[idx_view]
    v_masks  = test_masks_final[idx_view]
    
    v_metrics = compute_multilabel_metrics(v_probs, v_labels, v_masks, thresholds=best_thresholds_test)
    print(f"  {view_val:6s} (n={len(idx_view):,}): macro_AUC={v_metrics['macro_AUC']:.4f} | macro_F1={v_metrics['macro_F1']:.4f}")

# Subgrupo: raza
print("\n── AUC macro por RAZA ────────────────────────────────────────────────")
for race_name in RACE_MAP.keys():
    mask_race = df_test_reset["race"] == race_name
    idx_race  = mask_race[mask_race].index.values
    
    if len(idx_race) < 20:  # Mínimo 20 para AUC estable
        print(f"  {race_name:20s}: n={len(idx_race)} (insuficiente para AUC fiable)")
        continue
    
    r_probs  = test_probs_final[idx_race]
    r_labels = test_labels_final[idx_race]
    r_masks  = test_masks_final[idx_race]
    
    r_metrics = compute_multilabel_metrics(r_probs, r_labels, r_masks, thresholds=best_thresholds_test)
    print(f"  {race_name:20s} (n={len(idx_race):,}): macro_AUC={r_metrics['macro_AUC']:.4f} | macro_F1={r_metrics['macro_F1']:.4f}")

print("\n✅ Análisis de subgrupos completado.")
print("   Referencia TFM: comparar estos AUC para detectar posibles sesgos del modelo.")


---

## ✅ Resumen de lo que ha hecho este notebook

| Paso | Descripción |
|------|-------------|
| 1 | Carga de `train_clean.csv`, `val_clean.csv`, `test_clean.csv` con manejo de separador `;` |
| 2 | Dataset PyTorch con Masked BCE, política de incertidumbre, metadatos clínicos (13 features) |
| 3 | Pipeline de augmentación en 3 niveles: test / moderate / aggressive (albumentations) |
| 4 | ResNet-50 con pesos CheXpert (torchxrayvision) + módulo correlación 6×6 + rama metadatos |
| 5 | Masked BCE con `pos_weight` por etiqueta derivado del EDA (SPW 0.04 – 1.00) |
| 6 | Grid extenso de 12 hiperparámetros con Random Search sobre N=20 configuraciones |
| 7 | Nested CV: K_outer=5 (rendimiento) × K_inner=3 (selección) con Early Stopping (paciencia=5) |
| 8 | Búsqueda de umbral óptimo por etiqueta (F1-maximizing) sobre val interno |
| 9 | Evaluación final en test set oficial (una sola vez, nunca en tuning) |
| 10 | Análisis de equidad por género, raza y vista CXR |

## 📌 Próximos pasos para el TFM

1. **Ablation study**: comparar con `use_label_correlation=False` y `use_meta_branch=False` para cuantificar la contribución de cada módulo.
2. **Modelo ResNet1D para ECG**: arquitectura análoga pero con señales 1D (12 derivaciones).
3. **Modelo XGBoost para labs tabulares**: con los `labs_percentiles_train.npy` y flags de missingness.
4. **Late fusion stacking**: combinar las probabilidades de los 3 modelos base con un meta-clasificador (regresión logística multilabel) entrenado sobre el val set.
5. **Reporte en el TFM**: el número a citar es `mean_macro_AUC ± std` del Nested CV (no el AUC del test set, que es una estimación puntual potencialmente sobreoptimista).

## 📚 Referencias clave

- CheXpert: Irvin et al. (2019). *CheXpert: A large chest radiograph dataset with uncertainty labels and expert comparison.* AAAI 2019.
- torchxrayvision: Cohen et al. (2022). *TorchXRayVision: A library of chest X-ray datasets and models.* PMLR.
- ML-GCN: Chen et al. (2019). *Multi-label image recognition with graph convolutional networks.* CVPR 2019.
- Masked BCE: Rajpurkar et al. (2017). *CheXNet: Radiologist-level pneumonia detection on chest x-rays.* arXiv.
